# **HANDS - ON: FINE-TUNING CON LORA**

Una vez vista la masterclass ***Fine-tuning y Evaluación de Modelos***, se proporciona el siguiente ***Colab*** para ejecutar, en vivo, un fine-tuning real con LoRA sobre un modelo Llama ligero, y medir su mejora con una métrica objetiva.

A diferencia de los Temas anteriores, aquí no usamos Groq — Groq solo sirve para inferencia, no para entrenar modelos. Usamos **Hugging Face** (librerías `transformers` y `peft`) directamente sobre la GPU gratuita de Colab.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1cKZ_hCf231RE84FDvGkEiKMH6ZDkVZzH?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura, añadiendo un nombre asociado al token para guardarlo dentro de una variable y usarlo dentro del notebook. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

In [ ]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets
!pip install transformers peft accelerate trl --quiet

import torch
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("Sesión de Hugging Face iniciada correctamente.")

Sesión de Hugging Face iniciada correctamente.


### **CARGAR EL MODELO BASE**

Usamos un modelo Llama ligero (pocos parámetros) para que el fine-tuning corra en minutos sobre la GPU T4 gratuita de Colab, sin necesitar cuantización adicional.

In [ ]:
# Cargar el modelo base de Llama y su tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# variante oficial de Meta "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map='auto')
print('Modelo base cargado:', modelo_base)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


### **ANTES DEL FINE-TUNING: LÍNEA BASE**

Antes de ajustar nada, probamos el modelo base con un prompt de ejemplo para tener un punto de comparación. El modelo aún no conoce el tono ni el formato que le vamos a enseñar.

In [ ]:
# Definir una función para generar texto y probar el modelo base con un prompt de ejemplo
def generar_respuesta(modelo_a_usar, prompt, max_new_tokens=60):
    entrada = tokenizer(prompt, return_tensors='pt').to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada['input_ids'].shape[1]:]
    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)
    return texto_generado.split('\n')[0].strip()

prompt_prueba = 'Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente:'

respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

No, no puedes cambiar tu pedido.


### **PREPARAR LOS DATOS DE ENTRENAMIENTO**

El fine-tuning necesita ejemplos de entrada y salida que muestren el comportamiento que queremos enseñarle al modelo. Con pocos ejemplos (5 a 10) es suficiente para una demo — no es un dataset de producción.

In [ ]:
# Definir una lista de ejemplos (entrada -> respuesta esperada) y convertirla en dataset
from datasets import Dataset

ejemplos = [
    {"texto": "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente: Sí, puedes "
     "solicitar el cambio dentro de la primera hora escribiendo a soporte@tienda.com."},
    {"texto": "Cliente: ¿Cuánto tarda el reembolso?\nAgente: El reembolso se refleja en un plazo de 5 a 7 días hábiles."},
    {"texto": "Cliente: ¿Tienen envío el mismo día?\nAgente: Sí, disponible en zonas seleccionadas si el pedido se confirma antes de las 12:00."},
    {"texto": "Cliente: ¿Puedo pagar en el momento de la entrega?\nAgente: Sí, aceptamos pago contra entrega en efectivo o tarjeta."},
    {"texto": "Cliente: ¿Cómo rastreo mi paquete?\nAgente: Puedes rastrearlo con el número de guía en la sección 'Mis pedidos' de tu cuenta."},
]

dataset = Dataset.from_list(ejemplos)
dataset

Dataset({
    features: ['texto'],
    num_rows: 5
})

## **REALIZAR FINE-TUNING**

### **CONFIGURAR Y APLICAR LORA**

LoRA agrega matrices pequeñas entrenables sin tocar los pesos originales del modelo — por eso es tan ligero comparado con un fine-tuning completo.

In [ ]:
# Configurar LoRA (rango, alpha, módulos objetivo) y aplicarlo al modelo base
!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed

set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.0,
    task_type='CAUSAL_LM'
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()

# r más alto = más capacidad para aprender, pero también más parámetros entrenables

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


### **ENTRENAR CON LORA**

Con el dataset y LoRA ya configurados, ejecutamos el entrenamiento. La pérdida (*loss*) que reporta el entrenador es nuestra métrica objetiva: debería bajar a medida que el modelo aprende los ejemplos.

In [ ]:
# Configurar el entrenador (SFTTrainer) y ejecutar el fine-tuning
from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados",
    num_train_epochs=30,
    per_device_train_batch_size=5,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
print("Pérdida final:", resultado_entrenamiento.training_loss)

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '2.554', 'grad_norm': '1.737', 'learning_rate': '0.0002', 'entropy': '2.25', 'num_tokens': '229', 'mean_token_accuracy': '0.5', 'epoch': '1'}
{'loss': '2.512', 'grad_norm': '1.753', 'learning_rate': '0.0001933', 'entropy': '2.244', 'num_tokens': '458', 'mean_token_accuracy': '0.4955', 'epoch': '2'}
{'loss': '2.456', 'grad_norm': '1.926', 'learning_rate': '0.0001867', 'entropy': '2.234', 'num_tokens': '687', 'mean_token_accuracy': '0.5089', 'epoch': '3'}
{'loss': '2.387', 'grad_norm': '2.05', 'learning_rate': '0.00018', 'entropy': '2.223', 'num_tokens': '916', 'mean_token_accuracy': '0.5268', 'epoch': '4'}
{'loss': '2.317', 'grad_norm': '1.984', 'learning_rate': '0.0001733', 'entropy': '2.208', 'num_tokens': '1145', 'mean_token_accuracy': '0.5223', 'epoch': '5'}
{'loss': '2.246', 'grad_norm': '2.011', 'learning_rate': '0.0001667', 'entropy': '2.188', 'num_tokens': '1374', 'mean_token_accuracy': '0.5446', 'epoch': '6'}
{'loss': '2.17', 'grad_norm': '2.193', 'learning_rate': '0.0

### **DESPUÉS DEL FINE-TUNING: MEDIR LA MEJORA**

Compararemos la pérdida antes y después del entrenamiento como métrica objetiva.

In [ ]:
# Comparar la pérdidas

perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = resultado_entrenamiento.training_loss

print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

# trainer.state.log_history[0]['loss'] es la pérdida después del primer paso registrado,
# no la pérdida real del modelo sin ningún entrenamiento.

Pérdida al inicio del entrenamiento: 2.55
Pérdida final del entrenamiento: 1.77
Reducción: 31%


In [ ]:
# Referencia cualitativa (variación de sesión a sesión con un dataset chico)
respuesta_ajustada = generar_respuesta(modelo_lora, prompt_prueba)
print("\nRespuesta del modelo ajustado (referencia):\n", respuesta_ajustada)


Respuesta del modelo ajustado (referencia):
 Sleep, and the client.


**Nota:** el texto generado por el modelo ajustado puede variar de una ejecución a otra — con un dataset de solo 5 ejemplos y un learning rate alto (pensado para que el modelo aprenda rápido en pocos minutos), a veces la respuesta sale coherente y a veces sale con ruido. Esto es esperable en una demo de este tamaño, no un fallo del fine-tuning. La evidencia real de que el modelo aprendió es la **reducción de pérdida** (`perdida_inicial` vs. `perdida_final`), no el texto en sí — esa métrica sí es consistente ejecución tras ejecución, y es el criterio objetivo que estamos comprobando.

## **CHALLENGE: AJUSTE DE TONO CON LORA**

Una vez visto el ***Hands-On: Fine-tuning con LoRA***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se ajustará un modelo Llama ligero con un dataset propio para enseñarle un tono o formato de respuesta específico, comparando la pérdida **antes** y **después** del fine-tuning como métrica objetiva. En esta solución se usa como ejemplo un asistente de dudas frecuentes del propio curso.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Carga el modelo y define tu dataset:**

   * Instala las librerías, inicia sesión en Hugging Face con tu token y carga el modelo base junto con la función `generar_respuesta`.

   * Construye una lista llamada `ejemplos` con al menos 4 pares de entrada/respuesta que reflejen el tono o formato que quieres enseñarle al modelo, y conviértela en `dataset`.

In [ ]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets
!pip install transformers peft accelerate trl --quiet

import torch
from google.colab import userdata
from huggingface_hub import login

class HFManager:
    def __init__(self, key = userdata.get('HF_TOKEN')):
        try:
            self.token = key
            login(token=key)
            print("Sesión de Hugging Face iniciada correctamente.")
        except Exception as e:
            print(e)

hfm = HFManager()

Sesión de Hugging Face iniciada correctamente.


In [ ]:
# Cargar el modelo base de Llama y su tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging

class HFModelManager:
    def __init__(self, model_name):
        self.model_name = model_name
        self.model = None
        self.tokenizer = None

    def cargar(self):
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForCausalLM.from_pretrained(self.model_name, dtype=torch.float16, device_map='auto')
            print("Modelo base cargado:", self.model_name)
        except Exception as e:
            print(e)

hfmm = HFModelManager('TinyLlama/TinyLlama-1.1B-Chat-v1.0')
hfmm.cargar()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [ ]:
# Definir la funcion de generacion de texto
class HFTextGenerator:
    def __init__(self, modelo, tokenizer):
        self.modelo = modelo
        self.tokenizer = tokenizer

    def generar_respuesta(self, prompt, max_new_tokens=60):
        entrada = self.tokenizer(prompt,return_tensors="pt").to(self.modelo.device)
        salida = self.modelo.generate(**entrada,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=self.tokenizer.eos_token_id,no_repeat_ngram_size=3,)
        tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
        texto_generado = self.tokenizer.decode(tokens_nuevos,skip_special_tokens=True)

        return texto_generado.split("\n")[0].strip()

hftg = HFTextGenerator(hfmm.model, hfmm.tokenizer)
prompt_prueba = "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente:"

respuesta_base = hftg.generar_respuesta(prompt_prueba)
print(respuesta_base)

No, no puedes cambiar tu pedido.


**Nota**: para los ejemplos utilizaré un tono que asemeje el **español aureo**. Esto es más con fines didácticos para ver que tanto se puede modificar el comportamientos con **LoRA**

In [ ]:
# Definir la lista ejemplos y convertirla en dataset
from datasets import Dataset


class DatasetManager:

    def __init__(self, ejemplos):
        self.ejemplos = ejemplos
        self.dataset = None

    def crear_dataset(self):
        self.dataset = Dataset.from_list(self.ejemplos)
        return self.dataset


ejemplos = [
    {
        "texto": (
            "Cliente: ¿Puedo cambiar mi pedido después de haberlo pagado?\n"
            "Asistente: Bien podéis hacerlo, si no ha pasado más de una hora desde que se confirmó el pedido. "
            "Para tal menester, escribid a nuestro servicio de atención y procuraremos poner remedio al asunto."
        )
    },
    {
        "texto": (
            "Cliente: ¿Cuánto tarda en llegar el reembolso?\n"
            "Asistente: No suele hacerse esperar más de cinco a siete días hábiles. "
            "Tened, pues, un poco de paciencia, que el dinero, aunque tarda en ocasiones, "
            "suele hallar finalmente el camino de vuelta."
        )
    },
    {
        "texto": (
            "Cliente: ¿Cómo puedo saber dónde está mi paquete?\n"
            "Asistente: Si deseáis conocer por dónde anda vuestro pedido, acudid a la sección "
            "'Mis pedidos' e introducid el número de guía que os fue entregado. "
            "Allí podréis seguir sus pasos hasta que llegue a vuestras manos."
        )
    },
    {
        "texto": (
            "Cliente: ¿Puedo pagar cuando me entreguen el pedido?\n"
            "Asistente: Así es, buen señor. En aquellas tierras donde se halla disponible nuestro servicio "
            "de contraentrega, podéis satisfacer el importe en el momento de recibir vuestro pedido, "
            "ya sea en moneda o mediante tarjeta."
        )
    },
    {
        "texto": (
            "Cliente: He olvidado mi contraseña. ¿Qué puedo hacer?\n"
            "Asistente: No os aflijáis por tan pequeña desventura, que casi todo lo perdido puede ser hallado. "
            "En la pantalla de acceso escoged la opción 'Restablecer contraseña' y recibiréis en vuestro correo "
            "las instrucciones necesarias para recuperar el acceso."
        )
    },
    {
        "texto": (
            "Cliente: ¿Puedo cancelar mi suscripción?\n"
            "Asistente: Podéis hacerlo cuando fuere vuestro deseo. "
            "Basta con entrar en vuestra cuenta y buscar el apartado 'Configuración', "
            "donde hallaréis la opción para poner término a la suscripción."
        )
    },
    {
        "texto": (
            "Cliente: Mi pedido ha llegado con un producto equivocado.\n"
            "Asistente: Lamento sobremanera semejante yerro. "
            "Enviadnos una imagen del artículo que habéis recibido y pondremos diligencia en reparar el agravio, "
            "procurando que llegue a vuestras manos aquello que en verdad solicitasteis."
        )
    },
    {
        "texto": (
            "Cliente: ¿Realizan entregas durante los fines de semana?\n"
            "Asistente: En ciertos lugares, sí, aunque no en todos. "
            "Al llegar el momento de confirmar vuestro pedido, el sistema os mostrará si tal servicio se halla "
            "disponible en vuestra comarca."
        )
    },
    {
        "texto": (
            "Cliente: ¿Qué sucede si no estoy en casa cuando llegue el mensajero?\n"
            "Asistente: Si cuando llegare el mensajero no os hallare en casa, no deis el asunto por perdido. "
            "Podrá efectuarse un nuevo intento de entrega o concertarse otra fecha que resulte más conveniente."
        )
    },
    {
        "texto": (
            "Cliente: ¿Puedo solicitar una factura por mi compra?\n"
            "Asistente: Sin duda alguna. Entrad en vuestra cuenta y buscad el apartado destinado a las facturas. "
            "Allí podréis solicitarla, siempre que proporcionéis los datos fiscales que fueren menester."
        )
    },
]


dataset_manager = DatasetManager(ejemplos)

dataset = dataset_manager.crear_dataset()

dataset

Dataset({
    features: ['texto'],
    num_rows: 10
})

**2. Prueba el modelo base:** Genera una respuesta con el modelo sin ajustar para un prompt de prueba y guárdala en `respuesta_base`.

In [ ]:
# Probar el modelo base con un prompt de prueba y guardar el resultado en respuesta_base
prompt_prueba = "¿Qué puedo hacer si he olvidado mi contraseña?\nAsistente:"

# Generar y guardar la respuesta
respuesta_base = hftg.generar_respuesta(prompt_prueba)

print(respuesta_base)

¿Podrías proporcionarme la dirección de correo electrónico que utilizaste para registrarte en el sistema?


**3. Configura y aplica LoRA:** Fija una semilla con `set_seed` y define tu `LoraConfig` (rango, alpha, módulos objetivo, dropout en 0) y aplícalo al modelo base.

In [ ]:
# Configurar LoraConfig y aplicarlo al modelo base
from peft import LoraConfig, get_peft_model


class LoRAManager:

    def __init__(
        self,
        r=8,
        lora_alpha=16,
        lora_dropout=0.0,
        target_modules=None
    ):
        self.r = r
        self.lora_alpha = lora_alpha
        self.lora_dropout = lora_dropout
        self.target_modules = target_modules or ["q_proj", "v_proj"]

        self.config = None

    def configurar(self):
        self.config = LoraConfig(
            r=self.r,
            lora_alpha=self.lora_alpha,
            target_modules=self.target_modules,
            lora_dropout=self.lora_dropout,
            task_type="CAUSAL_LM"
        )

        return self.config

    def aplicar(self, modelo):
        if self.config is None:
            self.configurar()

        modelo_lora = get_peft_model(
            modelo,
            self.config
        )

        return modelo_lora

lora_manager = LoRAManager(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "v_proj"]
)

modelo_lora = lora_manager.aplicar(hfmm.model)
modelo_lora.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


**4. Entrena:** Configura el `SFTTrainer` con tu dataset y ejecuta el fine-tuning; guarda la pérdida final en `perdida_final`.

In [ ]:
# Configurar el Trainer con el dataset y ejecutar el fine-tuning; guardar la pérdida final en perdida_final
from trl import SFTTrainer, SFTConfig


class HFTrainer:

    def __init__(self, modelo, tokenizer, dataset):
        self.modelo = modelo
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.trainer = None

    def configurar(self):
        training_args = SFTConfig(
            output_dir="./resultados",
            num_train_epochs=30,
            per_device_train_batch_size=5,
            learning_rate=2e-4,
            logging_steps=1,
            dataset_text_field="texto",
            max_length=128,
            report_to="none",
        )

        self.trainer = SFTTrainer(
            model=self.modelo,
            train_dataset=self.dataset,
            args=training_args,
        )

        return self.trainer

    def entrenar(self):
        if self.trainer is None:
            self.configurar()

        resultado = self.trainer.train()

        perdida_final = resultado.training_loss

        return perdida_final

    def comparar_perdidas(self):
        logs = self.trainer.state.log_history

        perdidas = [
            log["loss"]
            for log in logs
            if "loss" in log
        ]

        perdida_inicial = perdidas[0]
        perdida_final = perdidas[-1]

        return perdida_inicial, perdida_final

trainer = HFTrainer(
    modelo_lora,
    hfmm.tokenizer,
    dataset
)

perdida_final = trainer.entrenar()

print("Pérdida final:", perdida_final)

Adding EOS to train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

{'loss': '2.92', 'grad_norm': '1.252', 'learning_rate': '0.0002', 'entropy': '2.259', 'num_tokens': '443', 'mean_token_accuracy': '0.4498', 'epoch': '0.5'}
{'loss': '2.824', 'grad_norm': '1.479', 'learning_rate': '0.0001967', 'entropy': '2.134', 'num_tokens': '831', 'mean_token_accuracy': '0.4465', 'epoch': '1'}
{'loss': '2.79', 'grad_norm': '1.534', 'learning_rate': '0.0001933', 'entropy': '2.199', 'num_tokens': '1236', 'mean_token_accuracy': '0.46', 'epoch': '1.5'}
{'loss': '2.825', 'grad_norm': '1.496', 'learning_rate': '0.00019', 'entropy': '2.225', 'num_tokens': '1662', 'mean_token_accuracy': '0.4489', 'epoch': '2'}
{'loss': '2.819', 'grad_norm': '1.685', 'learning_rate': '0.0001867', 'entropy': '2.27', 'num_tokens': '2088', 'mean_token_accuracy': '0.4466', 'epoch': '2.5'}
{'loss': '2.598', 'grad_norm': '1.67', 'learning_rate': '0.0001833', 'entropy': '2.163', 'num_tokens': '2493', 'mean_token_accuracy': '0.495', 'epoch': '3'}
{'loss': '2.635', 'grad_norm': '1.613', 'learning_rate

**5. Compara y concluye:** Calcula la reducción entre la pérdida inicial y `perdida_final` como métrica objetiva, y usa la respuesta generada por el modelo ya ajustado solo como referencia cualitativa.

In [ ]:
# Comparar la perdida inicial y final del entrenamiento como metrica objetiva
perdida_inicial, perdida_final = trainer.comparar_perdidas()

print(f"Pérdida inicial: {perdida_inicial:.4f}")
print(f"Pérdida final:   {perdida_final:.4f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

Pérdida inicial: 2.9205
Pérdida final:   1.3178
Reducción: 55%


In [ ]:
# Como referencia cualitativa (puede variar de sesión a sesión con un dataset tan chico):
respuesta_ajustada = generar_respuesta(modelo_lora, prompt_prueba)
print("\nRespuesta del modelo ajustado (referencia):\n", respuesta_ajustada)


Respuesta del modelo ajustado (referencia):
 No client client client.
